# LeetCode 845 — Longest Mountain in Array bằng Python

Link đề: https://leetcode.com/problems/longest-mountain-in-array/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Dynamic Programming, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def longestMountain(self, arr: List[int]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên). Đề có hai câu hỏi **Follow up**: giải **một lượt** và dùng **`O(1)` bộ nhớ** — notebook trả lời cả hai (Cách 3 và Cách 4). Oracle độc lập: kiểm tra **mọi mảng con** có phải "ngọn núi" theo đúng định nghĩa.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một mảng là **mảng ngọn núi** nếu độ dài `≥ 3` và tồn tại đỉnh `i` (`0 < i < len − 1`) sao cho **tăng chặt** tới `i` rồi **giảm chặt** sau `i`; tìm **mảng con liên tiếp** là ngọn núi **dài nhất** (hoặc `0`).
2. Nắm các công cụ Python: duyệt chỉ số, so sánh phần tử kề, hai con trỏ, mảng "độ dài đoạn tăng/giảm kết thúc/bắt đầu tại `i`" (quy hoạch động một chiều), vòng `while` nhảy cóc, `itertools` để vét cạn.
3. Đi từ **mở rộng từ mỗi đỉnh (`O(n²)`)** → **hai mảng `up`/`down` (`O(n)`, 3 lượt)** → **hai con trỏ nhảy cóc (một lượt, `O(1)` bộ nhớ)** → **bộ đếm `up`/`down` một lượt**.
4. Hiểu mấu chốt: ngọn núi là **một đoạn tăng chặt gắn liền một đoạn giảm chặt** tại đỉnh. Với mỗi đỉnh, độ dài ngọn núi lớn nhất qua nó là `up[i] + down[i] + 1`; và sau khi xử lý hết một ngọn núi ta có thể **nhảy** tới cuối sườn giảm vì các ngọn núi dài hơn không thể bắt đầu ở giữa.
5. Tránh các lỗi kinh điển: cho phép **đoạn phẳng** (`<=`), tính "núi" thiếu sườn giảm (hoặc sườn tăng), quên **cộng 1** (đếm số bước thay vì số phần tử), không nhảy cóc làm xấu đi thành `O(n²)`, và không reset bộ đếm khi gặp hai phần tử bằng nhau.

## 2. Đề bài & ví dụ

Một mảng `arr` là **mảng ngọn núi** (mountain array) khi và chỉ khi:

- `arr.length >= 3`, và
- tồn tại chỉ số `i` (từ 0) với `0 < i < arr.length − 1` sao cho:
  - `arr[0] < arr[1] < … < arr[i − 1] < arr[i]`
  - `arr[i] > arr[i + 1] > … > arr[arr.length − 1]`

Cho mảng số nguyên `arr`, trả về **độ dài của mảng con** (đoạn liên tiếp) **dài nhất là ngọn núi**. Trả về `0` nếu không có.

**Ví dụ 1:** `arr = [2,1,4,7,3,2,5]` → `5`. Ngọn núi lớn nhất là `[1,4,7,3,2]`.

**Ví dụ 2:** `arr = [2,2,2]` → `0`.

**Ràng buộc:** `1 <= arr.length <= 10^4`; `0 <= arr[i] <= 10^4`.

**Follow up:** (1) Có giải được trong **một lượt** duyệt? (2) Có giải được trong `O(1)` bộ nhớ?

In [1]:
import random
import time
from itertools import product
from typing import List

ex1 = [2, 1, 4, 7, 3, 2, 5]
ex2 = [2, 2, 2]
print("ex1:", ex1, "| bước tăng/giảm/bằng:", ["^" if b > a else "v" if b < a else "=" for a, b in zip(ex1, ex1[1:])])
print("ex2:", ex2, "| bước:", ["^" if b > a else "v" if b < a else "=" for a, b in zip(ex2, ex2[1:])])

ex1: [2, 1, 4, 7, 3, 2, 5] | bước tăng/giảm/bằng: ['v', '^', '^', 'v', 'v', '^']
ex2: [2, 2, 2] | bước: ['=', '=']


## 3. Kiến thức nền

### 3.1 Mảng con (subarray) và tăng/giảm **chặt**

**Mảng con** là các phần tử **liên tiếp** `arr[l..r]`. **Tăng chặt** nghĩa là `arr[k] < arr[k+1]` (bằng nhau không được). Một ngọn núi là chuỗi các bước `^ ^ … ^ v v … v` với **ít nhất một** bước `^` và **ít nhất một** bước `v`, và **không có bước `=`**.

In [2]:
def steps(a):
    return "".join("^" if y > x else "v" if y < x else "=" for x, y in zip(a, a[1:]))


for a in ([1, 4, 7, 3, 2], [2, 2, 2], [1, 2, 3], [3, 2, 1], [1, 3, 3, 2], [1, 2, 1]):
    print(f"{a} -> {steps(a)}")

[1, 4, 7, 3, 2] -> ^^vv
[2, 2, 2] -> ==
[1, 2, 3] -> ^^
[3, 2, 1] -> vv
[1, 3, 3, 2] -> ^=v
[1, 2, 1] -> ^v


### 3.2 Định nghĩa bằng chuỗi bước: biểu thức chính quy

Chuỗi bước của một ngọn núi có dạng `^+ v+` (một hoặc nhiều `^`, theo sau bởi một hoặc nhiều `v`). Ta dùng điều này để viết **hàm kiểm tra định nghĩa** `is_mountain` độc lập: cắt chuỗi bước tại ký tự `'v'` đầu tiên và kiểm tra hai nửa. Đây sẽ là **oracle** trong mục 12.

In [3]:
def is_mountain(a):
    """Kiểm tra ĐÚNG định nghĩa của đề: len >= 3, có đỉnh i với tăng chặt tới i rồi giảm chặt."""
    n = len(a)
    if n < 3:
        return False
    for i in range(1, n - 1):
        if all(a[k] < a[k + 1] for k in range(0, i)) and all(a[k] > a[k + 1] for k in range(i, n - 1)):
            return True
    return False


print([is_mountain(a) for a in ([1, 4, 7, 3, 2], [2, 2, 2], [1, 2, 3], [3, 2, 1], [1, 3, 3, 2], [1, 2, 1], [5])])

[True, False, False, False, False, True, False]


### 3.3 Độ dài đoạn tăng/giảm tại mỗi vị trí

Định nghĩa hai mảng:

- `up[i]` = số **bước tăng liên tiếp kết thúc tại `i`** (đi ngược từ `i` về trái bao nhiêu bước `^` liền nhau).
- `down[i]` = số **bước giảm liên tiếp bắt đầu tại `i`** (đi xuôi từ `i` sang phải bao nhiêu bước `v` liền nhau).

Nếu `up[i] > 0` và `down[i] > 0` thì `i` là đỉnh của một ngọn núi có độ dài `up[i] + down[i] + 1` (số bước cộng 1 = số phần tử). Hai mảng tính bằng quy hoạch động một chiều: `up[i] = up[i−1] + 1` nếu `arr[i−1] < arr[i]` ngược lại `0`; `down` tính từ phải sang.

In [4]:
arr = ex1
n = len(arr)
up = [0] * n
down = [0] * n
for i in range(1, n):
    if arr[i - 1] < arr[i]:
        up[i] = up[i - 1] + 1
for i in range(n - 2, -1, -1):
    if arr[i] > arr[i + 1]:
        down[i] = down[i + 1] + 1
print("arr :", arr)
print("up  :", up)
print("down:", down)
print("độ dài núi tại từng đỉnh:", [up[i] + down[i] + 1 if up[i] and down[i] else 0 for i in range(n)])

arr : [2, 1, 4, 7, 3, 2, 5]
up  : [0, 0, 1, 2, 0, 0, 1]
down: [1, 0, 0, 2, 1, 0, 0]
độ dài núi tại từng đỉnh: [0, 0, 0, 5, 0, 0, 0]


### 3.4 Hai con trỏ và "nhảy cóc"

Hai con trỏ `l` (chân núi) và `r` (đang di chuyển): từ `l` đi lên đỉnh, rồi đi xuống chân bên phải. Sau khi xử lý xong một ngọn núi (hoặc xác định không phải núi), ta đặt `l = r` (nhảy tới điểm kết thúc) thay vì `l + 1`: **tại sao an toàn** sẽ chứng minh ở mục 10. Nhờ nhảy cóc, mỗi phần tử được duyệt `O(1)` lần.

### 3.5 Bộ đếm một lượt

Thay vì lưu mảng, chỉ cần hai biến `up` (số bước tăng liên tiếp hiện tại) và `down` (số bước giảm liên tiếp sau đó) khi duyệt từ trái sang phải; reset khi chuỗi bước vỡ cấu trúc "núi". Đó là trạng thái của một **máy hữu hạn** với 3 trạng thái: *chưa lên*, *đang lên*, *đang xuống*.

## 4. Ý tưởng nền tảng / trực giác

Vẽ chuỗi bước của ví dụ 1 `[2,1,4,7,3,2,5]`: `v ^ ^ v v ^`.

```
chỉ số:  0  1  2  3  4  5  6
giá trị: 2  1  4  7  3  2  5
bước:      v  ^  ^  v  v  ^
```

Tìm các đoạn có dạng `^+ v+`: bước `^ ^ v v` (từ chỉ số 1 đến 5): 2 bước lên, 2 bước xuống → ngọn núi `[1,4,7,3,2]` có `2 + 2 + 1 = 5` phần tử. Bước cuối `^` (từ 2 lên 5) không có sườn xuống đi kèm, nên không phải núi. Đáp án `5`.

**Cách "ngây thơ" tự nhiên nhất:** với mỗi **đỉnh** (phần tử lớn hơn cả hai hàng xóm), mở rộng sang trái chừng nào còn tăng, sang phải chừng nào còn giảm. Mỗi đỉnh mất thời gian tỉ lệ với kích thước ngọn núi quanh nó; vì hai đỉnh khác nhau luôn cách nhau bởi một thung lũng nên sườn của chúng không chung nhau, và tổng công mở rộng thực tế là `O(n)` (mục 5 sẽ giải thích).

In [5]:
arr = ex1
for i in range(1, len(arr) - 1):
    if arr[i - 1] < arr[i] > arr[i + 1]:
        l = i
        while l > 0 and arr[l - 1] < arr[l]:
            l -= 1
        r = i
        while r < len(arr) - 1 and arr[r] > arr[r + 1]:
            r += 1
        print(f"đỉnh {i} (giá trị {arr[i]}): núi từ {l} tới {r} -> {arr[l:r + 1]} (độ dài {r - l + 1})")

đỉnh 3 (giá trị 7): núi từ 1 tới 5 -> [1, 4, 7, 3, 2] (độ dài 5)


## 5. Cách 1 — Brute Force: mở rộng từ mỗi đỉnh

**Ý tưởng:** với mỗi chỉ số `i` là **đỉnh** (`arr[i−1] < arr[i] > arr[i+1]`), mở rộng trái và phải như trên; độ dài `r − l + 1`; lấy lớn nhất. Dạng này dễ đúng và dễ hiểu.

In [6]:
def mountain_expand(arr):
    n = len(arr)
    best = 0
    for i in range(1, n - 1):
        if arr[i - 1] < arr[i] > arr[i + 1]:
            l = i
            while l > 0 and arr[l - 1] < arr[l]:
                l -= 1
            r = i
            while r < n - 1 and arr[r] > arr[r + 1]:
                r += 1
            best = max(best, r - l + 1)
    return best


print(mountain_expand(ex1), mountain_expand(ex2))
assert mountain_expand(ex1) == 5 and mountain_expand(ex2) == 0

5 0


**Độ phức tạp:** các ngọn núi quanh **hai đỉnh khác nhau** có sườn **không chung** (giữa hai đỉnh phải có một thung lũng), nên tổng công mở rộng là `O(n)`; nhưng với đầu vào tổng quát ta chứng minh được `O(n)`, bộ nhớ `O(1)`. (Nếu **không** kiểm tra "là đỉnh" rồi mở rộng cho mọi chỉ số thì sẽ `O(n²)` — xem lỗi 4 ở mục 13.)

Mặc dù đã `O(n)`, cách này **mở rộng hai lần** từ mỗi đỉnh (trái rồi phải) — hai lượt phụ. Ta xem các cách một lượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát:** sau khi xử lý xong một ngọn núi (hoặc quyết định tại vị trí `l` không có núi), **không cần xét lại** các vị trí nằm bên trong ngọn núi đó làm chân bắt đầu: một ngọn núi dài hơn không thể bắt đầu giữa một sườn tăng hay sườn giảm của ngọn núi vừa xét (bắt đầu từ giữa sườn thì ngắn hơn bắt đầu từ chân). Vì vậy:

- **Hai con trỏ nhảy cóc:** từ chân `l`, đi lên tới đỉnh `r` (nếu không đi lên được thì `l += 1`), rồi đi xuống tới `r'`; nếu cả lên và xuống đều thành công thì cập nhật đáp án; sau đó đặt `l = r'` (điểm cuối sườn xuống, cũng có thể là chân của ngọn núi kế tiếp).
- **Bộ đếm `up`/`down`:** duyệt một lần, duy trì số bước lên/xuống liên tiếp.
- **Mảng `up`/`down`:** tách thành hai lượt `O(n)` để dễ chứng minh.

Các cách trên đều `O(n)` thời gian; hai cách cuối cùng trả lời Follow up (**một lượt**, **`O(1)` bộ nhớ**).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** hai mảng `up`/`down`, **Cách 3** hai con trỏ nhảy cóc, **Cách 4** bộ đếm một lượt.

### Cách 2 — Hai mảng `up` và `down`

**Giải thích từng đoạn:**

1. `up[i]`: lượt trái → phải; `up[i] = up[i−1] + 1` nếu `arr[i−1] < arr[i]`, ngược lại `0`.
2. `down[i]`: lượt phải → trái; `down[i] = down[i+1] + 1` nếu `arr[i] > arr[i+1]`, ngược lại `0`.
3. Với mỗi `i` có `up[i] > 0` và `down[i] > 0`: ngọn núi qua đỉnh `i` dài `up[i] + down[i] + 1`.

In [7]:
def mountain_arrays(arr):
    n = len(arr)
    up = [0] * n
    down = [0] * n
    for i in range(1, n):
        if arr[i - 1] < arr[i]:
            up[i] = up[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if arr[i] > arr[i + 1]:
            down[i] = down[i + 1] + 1
    return max((up[i] + down[i] + 1 for i in range(n) if up[i] and down[i]), default=0)


print(mountain_arrays(ex1), mountain_arrays(ex2))
assert mountain_arrays(ex1) == 5 and mountain_arrays(ex2) == 0

5 0


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionArrays:
    def longestMountain(self, arr: List[int]) -> int:
        n = len(arr)
        up = [0] * n
        down = [0] * n
        for i in range(1, n):
            if arr[i - 1] < arr[i]:
                up[i] = up[i - 1] + 1
        for i in range(n - 2, -1, -1):
            if arr[i] > arr[i + 1]:
                down[i] = down[i + 1] + 1
        return max((up[i] + down[i] + 1 for i in range(n) if up[i] and down[i]), default=0)

**Độ phức tạp:** ba lượt `O(n)`; bộ nhớ `O(n)` (hai mảng).

### Cách 3 — Hai con trỏ nhảy cóc (một lượt, `O(1)` bộ nhớ)

**Giải thích từng đoạn:**

1. `l = 0` là chân ngọn núi đang xét; `ans = 0`.
2. Trong khi `l + 2 < n` (cần ít nhất 3 phần tử):
   - `r = l`. Nếu `arr[r] < arr[r+1]` (có thể đi lên): đi lên `while r + 1 < n and arr[r] < arr[r+1]: r += 1`.
   - Nếu đã lên ít nhất một bước (`r > l`) **và** `r < n − 1` **và** `arr[r] > arr[r+1]` (có thể đi xuống): đi xuống `while r + 1 < n and arr[r] > arr[r+1]: r += 1`; cập nhật `ans = max(ans, r − l + 1)`.
   - Nhảy: `l = max(r, l + 1)` (nếu `r == l` — không đi lên được — thì `l += 1`, vì phải tiến).

In [9]:
def mountain_two_pointers(arr):
    n = len(arr)
    ans = 0
    l = 0
    while l + 2 < n:
        r = l
        if arr[r] < arr[r + 1]:                              # có sườn tăng
            while r + 1 < n and arr[r] < arr[r + 1]:
                r += 1                                       # lên tới đỉnh
            if r < n - 1 and arr[r] > arr[r + 1]:            # có sườn giảm
                while r + 1 < n and arr[r] > arr[r + 1]:
                    r += 1                                   # xuống tới chân bên phải
                ans = max(ans, r - l + 1)
        l = max(r, l + 1)                                    # nhảy cóc tới cuối đoạn vừa duyệt
    return ans


print(mountain_two_pointers(ex1), mountain_two_pointers(ex2))
assert mountain_two_pointers(ex1) == 5 and mountain_two_pointers(ex2) == 0

5 0


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionTwoPointers:
    def longestMountain(self, arr: List[int]) -> int:
        n = len(arr)
        ans, l = 0, 0
        while l + 2 < n:
            r = l
            if arr[r] < arr[r + 1]:
                while r + 1 < n and arr[r] < arr[r + 1]:
                    r += 1
                if r < n - 1 and arr[r] > arr[r + 1]:
                    while r + 1 < n and arr[r] > arr[r + 1]:
                        r += 1
                    ans = max(ans, r - l + 1)
            l = max(r, l + 1)
        return ans

**Độ phức tạp:** mỗi vòng ngoài hoặc tăng `l` ít nhất `1`, hoặc nhảy tới `r` sau khi `r` đã đi qua một đoạn; tổng số bước của `r` qua mọi vòng là `O(n)` (mỗi phần tử được `r` duyệt tối đa hai lần). Thời gian `O(n)`, bộ nhớ `O(1)`.

### Cách 4 — Bộ đếm `up`/`down` một lượt

**Giải thích từng đoạn:** duyệt `i` từ `1` đến `n − 1`, so `arr[i−1]` với `arr[i]`, giữ hai bộ đếm `up`, `down`:

- Nếu đang xuống (`down > 0`) mà gặp bước **lên** (`arr[i−1] < arr[i]`), hoặc gặp bước **bằng** (`arr[i−1] == arr[i]`): ngọn núi hiện tại kết thúc → `up = down = 0`.
- Nếu bước **lên** (`arr[i−1] < arr[i]`): `up += 1`.
- Nếu bước **xuống** (`arr[i−1] > arr[i]`): `down += 1`.
- Nếu `up > 0` và `down > 0`: `ans = max(ans, up + down + 1)`.

Chú ý phải **reset trước** rồi mới tính tiếp bước hiện tại (vì bước lên sau khi xuống mở ra một ngọn núi **mới** bắt đầu bằng `up = 1`).

In [11]:
def mountain_counters(arr):
    up = down = 0
    ans = 0
    for i in range(1, len(arr)):
        if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
            up = down = 0                                    # núi cũ kết thúc (hoặc gặp đoạn phẳng)
        if arr[i - 1] < arr[i]:
            up += 1
        elif arr[i - 1] > arr[i]:
            down += 1
        if up and down:
            ans = max(ans, up + down + 1)
    return ans


print(mountain_counters(ex1), mountain_counters(ex2))
assert mountain_counters(ex1) == 5 and mountain_counters(ex2) == 0

5 0


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionCounters:
    def longestMountain(self, arr: List[int]) -> int:
        up = down = ans = 0
        for i in range(1, len(arr)):
            if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
                up = down = 0
            if arr[i - 1] < arr[i]:
                up += 1
            elif arr[i - 1] > arr[i]:
                down += 1
            if up and down:
                ans = max(ans, up + down + 1)
        return ans

**Độ phức tạp:** một lượt `O(n)` thời gian, `O(1)` bộ nhớ. Đây là câu trả lời ngắn gọn nhất cho cả hai câu hỏi Follow up.

## 8. TRACE từng bước

**Trace Cách 4 (bộ đếm)** trên `[2,1,4,7,3,2,5]`:

| `i` | `arr[i−1] → arr[i]` | bước | `up` | `down` | `up+down+1` (nếu cả hai > 0) | `ans` |
|---|---|---|---|---|---|---|
| 1 | `2 → 1` | xuống | 0 | 1 | — | 0 |
| 2 | `1 → 4` | lên (đang `down > 0` → reset, rồi `up = 1`) | 1 | 0 | — | 0 |
| 3 | `4 → 7` | lên | 2 | 0 | — | 0 |
| 4 | `7 → 3` | xuống | 2 | 1 | 4 | 4 |
| 5 | `3 → 2` | xuống | 2 | 2 | 5 | 5 |
| 6 | `2 → 5` | lên (đang `down > 0` → reset, rồi `up = 1`) | 1 | 0 | — | 5 |

Đáp án `5`. Hàm trace in từng bước:

In [13]:
def trace_counters(arr):
    up = down = ans = 0
    print("arr =", arr)
    for i in range(1, len(arr)):
        a, b = arr[i - 1], arr[i]
        note = []
        if (down and a < b) or a == b:
            note.append("reset (núi cũ kết thúc hoặc đoạn phẳng)")
            up = down = 0
        if a < b:
            up += 1
            kind = "lên"
        elif a > b:
            down += 1
            kind = "xuống"
        else:
            kind = "bằng"
        cur = up + down + 1 if up and down else None
        if cur:
            ans = max(ans, cur)
        print(f"i={i}: {a} -> {b} ({kind:5s}) up={up} down={down} | độ dài núi hiện tại {cur if cur else '-'} | ans={ans}" + (" | " + "; ".join(note) if note else ""))
    print("=> đáp án", ans)
    return ans


assert trace_counters(ex1) == 5
print()
assert trace_counters([1, 3, 3, 2, 1, 4, 2]) == 3

arr = [2, 1, 4, 7, 3, 2, 5]
i=1: 2 -> 1 (xuống) up=0 down=1 | độ dài núi hiện tại - | ans=0
i=2: 1 -> 4 (lên  ) up=1 down=0 | độ dài núi hiện tại - | ans=0 | reset (núi cũ kết thúc hoặc đoạn phẳng)
i=3: 4 -> 7 (lên  ) up=2 down=0 | độ dài núi hiện tại - | ans=0
i=4: 7 -> 3 (xuống) up=2 down=1 | độ dài núi hiện tại 4 | ans=4
i=5: 3 -> 2 (xuống) up=2 down=2 | độ dài núi hiện tại 5 | ans=5
i=6: 2 -> 5 (lên  ) up=1 down=0 | độ dài núi hiện tại - | ans=5 | reset (núi cũ kết thúc hoặc đoạn phẳng)
=> đáp án 5

arr = [1, 3, 3, 2, 1, 4, 2]
i=1: 1 -> 3 (lên  ) up=1 down=0 | độ dài núi hiện tại - | ans=0
i=2: 3 -> 3 (bằng ) up=0 down=0 | độ dài núi hiện tại - | ans=0 | reset (núi cũ kết thúc hoặc đoạn phẳng)
i=3: 3 -> 2 (xuống) up=0 down=1 | độ dài núi hiện tại - | ans=0
i=4: 2 -> 1 (xuống) up=0 down=2 | độ dài núi hiện tại - | ans=0
i=5: 1 -> 4 (lên  ) up=1 down=0 | độ dài núi hiện tại - | ans=0 | reset (núi cũ kết thúc hoặc đoạn phẳng)
i=6: 4 -> 2 (xuống) up=1 down=1 | độ dài núi hiện tại 3 | a

**Trace Cách 3 (hai con trỏ nhảy cóc)** trên cùng mảng: `l = 0`: `arr[0] = 2 > arr[1] = 1` → không đi lên được: `r = 0`, `l = max(0, 1) = 1`. `l = 1`: lên `1→4→7` (`r = 3`), có sườn giảm `7 > 3`: xuống `7→3→2` (`r = 5`), `ans = 5 − 1 + 1 = 5`; `l = max(5, 2) = 5`. `l = 5`: `l + 2 = 7 ≥ n = 7` → dừng. Hàm trace:

In [14]:
def trace_two_pointers(arr):
    n = len(arr)
    ans, l = 0, 0
    print("arr =", arr)
    while l + 2 < n:
        r = l
        info = f"l={l}: "
        if arr[r] < arr[r + 1]:
            while r + 1 < n and arr[r] < arr[r + 1]:
                r += 1
            info += f"lên tới đỉnh r={r} ({arr[r]})"
            if r < n - 1 and arr[r] > arr[r + 1]:
                while r + 1 < n and arr[r] > arr[r + 1]:
                    r += 1
                ans = max(ans, r - l + 1)
                info += f", xuống tới r={r}: núi {arr[l:r + 1]} độ dài {r - l + 1}"
            else:
                info += ", không có sườn giảm -> không phải núi"
        else:
            info += "không đi lên được"
        nl = max(r, l + 1)
        print(f"{info} | nhảy l -> {nl} | ans={ans}")
        l = nl
    print("=> đáp án", ans)
    return ans


assert trace_two_pointers(ex1) == 5
print()
assert trace_two_pointers([0, 1, 2, 1, 0, 1, 2, 3, 2, 1, 0]) == 7

arr = [2, 1, 4, 7, 3, 2, 5]
l=0: không đi lên được | nhảy l -> 1 | ans=0
l=1: lên tới đỉnh r=3 (7), xuống tới r=5: núi [1, 4, 7, 3, 2] độ dài 5 | nhảy l -> 5 | ans=5
=> đáp án 5

arr = [0, 1, 2, 1, 0, 1, 2, 3, 2, 1, 0]
l=0: lên tới đỉnh r=2 (2), xuống tới r=4: núi [0, 1, 2, 1, 0] độ dài 5 | nhảy l -> 4 | ans=5
l=4: lên tới đỉnh r=7 (3), xuống tới r=10: núi [0, 1, 2, 3, 2, 1, 0] độ dài 7 | nhảy l -> 10 | ans=7
=> đáp án 7


## 9. Code nộp LeetCode

Chọn **bộ đếm `up`/`down`** (một lượt, `O(1)` bộ nhớ, ngắn). Giữ nguyên `class Solution`, `longestMountain(self, arr)`:

In [15]:
from typing import List


class Solution:
    def longestMountain(self, arr: List[int]) -> int:
        up = down = ans = 0                       # số bước lên / xuống liên tiếp hiện tại
        for i in range(1, len(arr)):
            if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
                up = down = 0                     # núi cũ kết thúc, hoặc gặp đoạn phẳng
            if arr[i - 1] < arr[i]:
                up += 1
            elif arr[i - 1] > arr[i]:
                down += 1
            if up and down:                       # có cả sườn lên và sườn xuống
                ans = max(ans, up + down + 1)     # +1: số phần tử = số bước + 1
        return ans


print(Solution().longestMountain([2, 1, 4, 7, 3, 2, 5]), Solution().longestMountain([2, 2, 2]))   # 5 0
assert Solution().longestMountain(ex1) == 5 and Solution().longestMountain(ex2) == 0

5 0


## 10. Vì sao đúng

**(a) Định nghĩa qua bước.** Một mảng con `arr[l..r]` là ngọn núi khi và chỉ khi chuỗi bước của nó là `^^…^ vv…v` với số `^ ≥ 1` và số `v ≥ 1` (nghĩa là tăng chặt rồi giảm chặt, đỉnh nằm ở chỗ chuyển từ `^` sang `v`, độ dài `= số bước + 1 ≥ 3`).

**(b) Cách 2.** Với mỗi `i`, `up[i]` là số bước `^` liên tiếp kết thúc tại `i`, `down[i]` là số bước `v` liên tiếp bắt đầu tại `i`. Ngọn núi dài nhất nhận `i` làm đỉnh có `up[i]` bước `^` ngay trước và `down[i]` bước `v` ngay sau, tức dài `up[i] + down[i] + 1` khi cả hai `> 0` (nếu thiếu một phía thì không phải núi). Lấy `max` theo mọi đỉnh `i` ra ngọn núi dài nhất.

**(c) Cách 4 (bộ đếm).** Bất biến sau mỗi bước `i`: nếu `up > 0` và `down > 0` thì `arr[i − up − down .. i]` là ngọn núi **đang kết thúc tại `i`**, dài nhất có thể kết thúc ở `i`; nếu `down == 0`, `up` là độ dài sườn tăng đang xây (không phải núi); nếu cả hai `0` thì chưa có sườn. Chuyển trạng thái: bước `=` phá mọi cấu trúc (không có sườn nào chứa hai phần tử bằng nhau) → reset; bước `^` sau khi đã có `v` (`down > 0`): ngọn núi cũ không thể kéo dài (đã xuống rồi lên), và một sườn tăng **mới** bắt đầu → reset rồi `up = 1`; bước `^` khi `down = 0` kéo dài sườn tăng; bước `v` kéo dài sườn xuống (hoặc bắt đầu nó nếu `up > 0`; nếu `up = 0` thì `down` tăng mà không có núi: giá trị `down` lớn nhưng `up = 0` nên `up and down` sai — ta chỉ báo núi khi cả hai `> 0`).

**(d) Cách 3 (nhảy cóc) không bỏ sót.** Giả sử sau khi xử lý từ chân `l`, đoạn `[l, r]` là: (i) một ngọn núi hoàn chỉnh (lên rồi xuống), hoặc (ii) lên rồi dừng (không có sườn giảm, vì `arr[r] ≤ arr[r+1]`... thực ra `arr[r] == arr[r+1]` hoặc hết mảng), hoặc (iii) không lên được (`arr[l] ≥ arr[l+1]`). Một ngọn núi bắt đầu tại `l' ∈ (l, r)` (nằm giữa) có cùng đỉnh/sườn xuống với ngọn núi bắt đầu từ chân `l` trong trường hợp (i) nên ngắn hơn; trong trường hợp (ii) mọi `l' ∈ (l, r)` nằm trong sườn tăng kết thúc bằng bước không phải `v` (đã không có sườn giảm), nên không có núi bắt đầu từ `l'`; trong (iii) `r = l` và ta chỉ tăng `l` thêm `1`. Tại điểm `r` (cuối sườn xuống), `r` có thể là chân của một núi mới (nếu tiếp theo là bước `^`), nên ta đặt `l = r` (không phải `r + 1`).

Ta kiểm chứng bằng máy: 4 cách khớp **oracle kiểm tra mọi mảng con theo định nghĩa** trên **mọi** mảng nhỏ (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mở rộng từ mỗi đỉnh | `O(n)` (mỗi sườn duyệt không quá hai lần) | `O(1)` | đơn giản nhất |
| 2. Hai mảng `up`/`down` | `O(n)` (3 lượt) | `O(n)` | dễ chứng minh |
| 3. Hai con trỏ nhảy cóc | `O(n)` | `O(1)` | **một lượt**, `O(1)` bộ nhớ |
| 4. Bộ đếm `up`/`down` | `O(n)` | `O(1)` | **một lượt**, ngắn gọn nhất |

Đo thực tế (`n = 10^4`, các dạng mảng):

In [16]:
def bench(fn, a, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(a)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(845)
n = 10 ** 4
shapes = {
    "ngẫu nhiên": [rng.randint(0, 10 ** 4) for _ in range(n)],
    "một ngọn núi lớn": list(range(5000)) + list(range(5000, 0, -1)),
    "răng cưa": [i % 2 * 5 for i in range(n)],
    "tăng dần": list(range(n)),
    "toàn bằng nhau": [7] * n,
}
for name, a in shapes.items():
    res = [fn(a) for fn in (mountain_expand, mountain_arrays, mountain_two_pointers, mountain_counters)]
    assert len(set(res)) == 1
    print(f"{name:18s} đáp án {res[0]:5d} | " + " | ".join(f"{k} {bench(fn, a) * 1000:6.2f} ms" for k, fn in [("expand", mountain_expand), ("arrays", mountain_arrays), ("2ptr", mountain_two_pointers), ("count", mountain_counters)]))

ngẫu nhiên         đáp án     9 | expand   3.03 ms | arrays   2.91 ms | 2ptr   2.95 ms | count   2.35 ms


một ngọn núi lớn   đáp án 10000 | expand   2.03 ms | arrays   2.30 ms | 2ptr   0.72 ms | count   1.82 ms


răng cưa           đáp án     3 | expand   2.75 ms | arrays   2.49 ms | 2ptr   2.86 ms | count   1.98 ms


tăng dần           đáp án     0 | expand   1.13 ms | arrays   2.12 ms | 2ptr   0.90 ms | count   1.32 ms
toàn bằng nhau     đáp án     0 | expand   0.60 ms | arrays   1.40 ms | 2ptr   1.25 ms | count   1.84 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn** mọi mảng nhỏ với oracle "kiểm tra mọi mảng con theo định nghĩa", (E) ngẫu nhiên có seed, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [17]:
def oracle(arr):
    """Oracle theo định nghĩa: thử MỌI mảng con và kiểm tra is_mountain; lấy độ dài lớn nhất."""
    n = len(arr)
    best = 0
    for l in range(n):
        for r in range(l + 2, n):
            if r - l + 1 > best and is_mountain(arr[l:r + 1]):
                best = r - l + 1
    return best


IMPLS = {
    "expand": mountain_expand,
    "arrays": mountain_arrays,
    "two_pointers": mountain_two_pointers,
    "counters": mountain_counters,
    "Solution": lambda a: Solution().longestMountain(list(a)),
    "SolutionArrays": lambda a: SolutionArrays().longestMountain(list(a)),
    "SolutionTwoPointers": lambda a: SolutionTwoPointers().longestMountain(list(a)),
    "SolutionCounters": lambda a: SolutionCounters().longestMountain(list(a)),
}


def check(arr, want=None, names=None):
    if want is None:
        want = oracle(arr)
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr))
        assert got == want, (name, arr, got, want)


check(ex1, 5)
check(ex2, 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng; oracle cũng cho", oracle(ex1), oracle(ex2))

ví dụ 1, 2: cả 8 cài đặt đúng; oracle cũng cho 5 0


### B. Edge case

- Độ dài `1`, `2` → `0`.
- Chỉ tăng; chỉ giảm; toàn bằng nhau → `0`.
- Núi tối thiểu `[1, 2, 1]` → `3`.
- Đỉnh phẳng `[1, 3, 3, 2]` → `0` (đoạn phẳng phá núi).
- Hai ngọn núi sát nhau chia sẻ chân: `[1,2,1,2,1]` → `3`.
- Núi ở đầu mảng, ở cuối mảng.
- Hai núi khác độ dài, núi dài ở giữa.
- Dốc rất dài một phía (`[0..4999, 4999..0]` không đối xứng).

In [18]:
cases = [
    ([5], 0), ([5, 6], 0), ([1, 2, 3, 4], 0), ([4, 3, 2, 1], 0), ([3, 3, 3, 3], 0),
    ([1, 2, 1], 3), ([1, 3, 3, 2], 0), ([1, 2, 1, 2, 1], 3), ([1, 2, 3, 2, 1, 2, 1], 5),
    ([2, 1, 4, 7, 3, 2, 5], 5), ([0, 1, 2, 3, 4, 5, 4, 3, 2, 1, 0], 11),
    ([1, 2, 3, 3, 2, 1], 0), ([1, 3, 2, 2, 1], 3), ([2, 3, 2, 3, 4, 3, 2, 1], 6),
    ([1, 1, 2, 3, 2, 2, 1], 4), ([9, 8, 9, 10, 9, 8, 7], 6),
]
for arr, want in cases:
    check(arr, want)
    assert oracle(arr) == want, (arr, oracle(arr), want)
print(len(cases), "edge case: tất cả cài đặt và oracle khớp đáp án viết tay")

16 edge case: tất cả cài đặt và oracle khớp đáp án viết tay


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_plateau_allowed(arr):                     # cho phép bằng nhau (<=, >=)
    n = len(arr)
    best = 0
    for i in range(1, n - 1):
        if arr[i - 1] <= arr[i] >= arr[i + 1] and arr[i - 1] < arr[i] or arr[i] > arr[i + 1] and arr[i - 1] <= arr[i]:
            l = i
            while l > 0 and arr[l - 1] <= arr[l]:
                l -= 1
            r = i
            while r < n - 1 and arr[r] >= arr[r + 1]:
                r += 1
            best = max(best, r - l + 1)
    return best if best >= 3 else 0


def bug_no_descent(arr):                          # tính cả sườn tăng không có sườn giảm
    up = ans = 0
    for i in range(1, len(arr)):
        up = up + 1 if arr[i - 1] < arr[i] else 0
        ans = max(ans, up + 1) if up else ans
    return ans if ans >= 3 else 0


def bug_forget_plus_one(arr):                     # trả số bước thay vì số phần tử
    up = down = ans = 0
    for i in range(1, len(arr)):
        if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
            up = down = 0
        if arr[i - 1] < arr[i]:
            up += 1
        elif arr[i - 1] > arr[i]:
            down += 1
        if up and down:
            ans = max(ans, up + down)             # SAI: thiếu +1
    return ans


def bug_no_reset_equal(arr):                      # không reset khi gặp bằng nhau
    up = down = ans = 0
    for i in range(1, len(arr)):
        if down and arr[i - 1] < arr[i]:
            up = down = 0
        if arr[i - 1] < arr[i]:
            up += 1
        elif arr[i - 1] > arr[i]:
            down += 1
        if up and down:
            ans = max(ans, up + down + 1)
    return ans


def bug_no_reset_after_down(arr):                 # không reset khi lên sau xuống (núi mới)
    up = down = ans = 0
    for i in range(1, len(arr)):
        if arr[i - 1] == arr[i]:
            up = down = 0
        if arr[i - 1] < arr[i]:
            up += 1
        elif arr[i - 1] > arr[i]:
            down += 1
        if up and down:
            ans = max(ans, up + down + 1)
    return ans


BUGS = [bug_plateau_allowed, bug_no_descent, bug_forget_plus_one, bug_no_reset_equal, bug_no_reset_after_down]
rng = random.Random(1)
tests = [[rng.randint(0, 3) for _ in range(rng.randint(1, 9))] for _ in range(1500)]
for bug in BUGS:
    bad = next((t for t in tests if bug(t) != oracle(t)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên {bad}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_plateau_allowed        sai trên [3, 3, 3, 1, 0, 3, 0, 3]: ra 5, đúng 3
bug_no_descent             sai trên [0, 2, 0]: ra 0, đúng 3
bug_forget_plus_one        sai trên [0, 2, 0]: ra 2, đúng 3
bug_no_reset_equal         sai trên [0, 3, 1, 3, 0, 1, 3, 3, 1]: ra 4, đúng 3
bug_no_reset_after_down    sai trên [3, 3, 3, 1, 0, 3, 0, 3]: ra 6, đúng 3


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `1..8` với giá trị trong `{0, 1, 2}` (`3 + 9 + … + 6561 = 9840` mảng) và mọi mảng độ dài `1..6` với giá trị trong `{0, 1, 2, 3}` (`5460` mảng) — đủ để có đoạn phẳng, nhiều đỉnh, núi dài. Mọi cài đặt phải khớp oracle (kiểm mọi mảng con theo định nghĩa).

In [20]:
t0 = time.perf_counter()
cnt = 0
dist = {}
for values, max_len in ((3, 8), (4, 6)):
    for L in range(1, max_len + 1):
        for t in product(range(values), repeat=L):
            a = list(t)
            want = oracle(a)
            for name in IMPLS:
                assert IMPLS[name](a) == want, (name, a)
            dist[want] = dist.get(want, 0) + 1
            cnt += 1
print(cnt, "mảng vét cạn: 8 cài đặt khớp oracle; phân bố đáp án:", dict(sorted(dist.items())), f"; {time.perf_counter() - t0:.1f} s")

15300 mảng vét cạn: 8 cài đặt khớp oracle; phân bố đáp án: {0: 4797, 3: 7373, 4: 2850, 5: 274, 6: 6} ; 0.5 s


### E. Ngẫu nhiên có seed, đúng ràng buộc

Mảng ngẫu nhiên độ dài `1..2000` với khoảng giá trị khác nhau (nhiều đoạn phẳng khi khoảng nhỏ), thêm các mảng **gieo** nhiều ngọn núi; oracle chạy cho `n ≤ 60`, còn lớn hơn thì so các cách với nhau và với cách mở rộng từ đỉnh.

In [21]:
def plant_mountains(rng, n):
    a = []
    while len(a) < n:
        up_len, down_len = rng.randint(1, 6), rng.randint(1, 6)
        base = rng.randint(0, 20)
        a += list(range(base, base + up_len)) + list(range(base + up_len, base + up_len - down_len, -1))
        if rng.random() < 0.3:
            a += [a[-1]] * rng.randint(1, 2)                  # đoạn phẳng giữa hai núi
    return a[:n]


rng = random.Random(20240845)
count = 0
for _ in range(1500):
    n = rng.randint(1, 60)
    hi = rng.choice([1, 2, 4, 10, 10 ** 4])
    a = [rng.randint(0, hi) for _ in range(n)] if rng.random() < 0.6 else plant_mountains(rng, n)
    check(a, oracle(a))
    count += 1
for _ in range(300):
    n = rng.randint(100, 2000)
    a = [rng.randint(0, rng.choice([1, 3, 10, 10 ** 4])) for _ in range(n)] if rng.random() < 0.5 else plant_mountains(rng, n)
    want = mountain_expand(a)
    check(a, want)
    count += 1
print(count, "mảng ngẫu nhiên (cả n tới 2000): mọi cài đặt trùng oracle / cách mở rộng")

# tính chất: đảo ngược mảng không đổi đáp án; thêm số vào hai đầu không làm đáp án giảm; cộng hằng không đổi
for _ in range(500):
    a = [rng.randint(0, 6) for _ in range(rng.randint(1, 30))]
    base = Solution().longestMountain(a)
    assert Solution().longestMountain(a[::-1]) == base
    assert Solution().longestMountain([rng.randint(0, 9)] + a + [rng.randint(0, 9)]) >= base
    assert Solution().longestMountain([v + 5 for v in a]) == base
print("tính chất: đảo ngược, thêm hai đầu không giảm, cộng hằng — OK")

1800 mảng ngẫu nhiên (cả n tới 2000): mọi cài đặt trùng oracle / cách mở rộng
tính chất: đảo ngược, thêm hai đầu không giảm, cộng hằng — OK


### F. Giới hạn lớn nhất

`n = 10^4`: ngọn núi khổng lồ (đáp án `10^4`), núi nhỏ lặp, tăng dần (`0`), giảm dần (`0`), toàn bằng nhau (`0`), đỉnh phẳng ở giữa một núi lớn.

In [22]:
n = 10 ** 4
cases = {
    "núi khổng lồ (tăng 5000 rồi giảm 5000)": (list(range(5000)) + list(range(5000, 0, -1)), 10 ** 4),
    "tăng dần": (list(range(n)), 0),
    "giảm dần": (list(range(n, 0, -1)), 0),
    "toàn bằng nhau": ([3] * n, 0),
    "núi lặp 3": ([1, 2, 1] * (n // 3), 3),
    "răng cưa lên-xuống": ([i % 2 for i in range(n)], 3),
    "đỉnh phẳng giữa núi lớn": (list(range(4999)) + [5000, 5000] + list(range(4998, 0, -1)), 0),
    "núi dài nhất ở cuối": ([1, 2, 1] * 100 + list(range(1, 3000)) + list(range(3000, 0, -1)), 5999),
}
for name, (a, want) in cases.items():
    got = mountain_expand(a)
    assert got == want, (name, got, want)
    for impl in IMPLS:
        assert IMPLS[impl](a) == want, (name, impl)
    print(f"{name:40s}: {want:6d} (8 cài đặt khớp)")

núi khổng lồ (tăng 5000 rồi giảm 5000)  :  10000 (8 cài đặt khớp)
tăng dần                                :      0 (8 cài đặt khớp)
giảm dần                                :      0 (8 cài đặt khớp)
toàn bằng nhau                          :      0 (8 cài đặt khớp)
núi lặp 3                               :      3 (8 cài đặt khớp)
răng cưa lên-xuống                      :      3 (8 cài đặt khớp)
đỉnh phẳng giữa núi lớn                 :      0 (8 cài đặt khớp)
núi dài nhất ở cuối                     :   5999 (8 cài đặt khớp)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cho phép đoạn phẳng (dùng `<=` thay `<`)

Đề nói **tăng chặt** và **giảm chặt**. `[1, 3, 3, 2]` có đỉnh "phẳng" (`3, 3`): **không** phải ngọn núi.

In [23]:
a = [1, 3, 3, 2]
print("bản cho phép bằng nhau:", bug_plateau_allowed(a), "| đúng:", Solution().longestMountain(a))
a = [1, 2, 2, 3, 2, 1]
print("ví dụ khác:", bug_plateau_allowed(a), "| đúng:", Solution().longestMountain(a), "(núi thật chỉ là [2,3,2,1])")
assert bug_plateau_allowed([1, 3, 3, 2]) != 0 and Solution().longestMountain([1, 3, 3, 2]) == 0

bản cho phép bằng nhau: 4 | đúng: 0
ví dụ khác: 6 | đúng: 4 (núi thật chỉ là [2,3,2,1])


**Vì sao sai:** bằng nhau phá cấu trúc tăng/giảm chặt. **Đúng:** so sánh chặt `<`, `>`; khi gặp `==` thì reset (Cách 4) hoặc dừng mở rộng (Cách 1).

### Lỗi 2: tính sườn tăng không có sườn giảm là núi

Đoạn tăng dần `[1,2,3,4]` **không** phải núi vì không có phần giảm. Phải yêu cầu **cả hai** `up > 0` và `down > 0`.

In [24]:
print("bản thiếu sườn giảm trên [1,2,3,4]:", bug_no_descent([1, 2, 3, 4]), "| đúng:", Solution().longestMountain([1, 2, 3, 4]))
print("trên ví dụ 1:", bug_no_descent(ex1), "| đúng:", Solution().longestMountain(ex1))
assert bug_no_descent([1, 2, 3, 4]) == 4 and Solution().longestMountain([1, 2, 3, 4]) == 0

bản thiếu sườn giảm trên [1,2,3,4]: 4 | đúng: 0
trên ví dụ 1: 3 | đúng: 5


**Vì sao sai:** ngọn núi cần đỉnh **ở giữa** (`0 < i < len − 1`), tức cả hai sườn không rỗng. **Đúng:** `if up and down`.

### Lỗi 3: quên cộng 1 (đếm bước thay vì phần tử)

`up` và `down` đếm **số bước**; số phần tử là số bước `+ 1`. Ví dụ `[1,2,1]` có `1` bước lên, `1` bước xuống, nhưng `3` phần tử.

In [25]:
print("bản thiếu +1 trên [1,2,1]:", bug_forget_plus_one([1, 2, 1]), "| đúng:", Solution().longestMountain([1, 2, 1]))
print("trên ví dụ 1:", bug_forget_plus_one(ex1), "| đúng:", Solution().longestMountain(ex1))
assert bug_forget_plus_one(ex1) == 4 and Solution().longestMountain(ex1) == 5

bản thiếu +1 trên [1,2,1]: 2 | đúng: 3
trên ví dụ 1: 4 | đúng: 5


**Vì sao sai:** `n` phần tử tạo `n − 1` bước. **Đúng:** `up + down + 1`.

### Lỗi 4: không reset bộ đếm đúng lúc

Hai tình huống cần reset: (a) gặp bước `=`; (b) đang xuống (`down > 0`) rồi gặp bước lên — ngọn núi cũ đã kết thúc và một ngọn mới bắt đầu bằng `up = 1`. Bỏ một trong hai làm "nối" nhầm các ngọn núi.

In [26]:
a = [1, 2, 1, 1, 0]
print(f"{a}: không reset khi '=': {bug_no_reset_equal(a)} | đúng: {Solution().longestMountain(a)} (đoạn phẳng 1,1 chia đôi sườn xuống)")
a = [1, 3, 2, 3, 4, 1]
print(f"{a}: không reset khi lên sau xuống: {bug_no_reset_after_down(a)} | đúng: {Solution().longestMountain(a)}")
assert bug_no_reset_equal([1, 2, 1, 1, 0]) == 4 and Solution().longestMountain([1, 2, 1, 1, 0]) == 3
assert bug_no_reset_after_down([1, 3, 2, 3, 4, 1]) != Solution().longestMountain([1, 3, 2, 3, 4, 1])

[1, 2, 1, 1, 0]: không reset khi '=': 4 | đúng: 3 (đoạn phẳng 1,1 chia đôi sườn xuống)
[1, 3, 2, 3, 4, 1]: không reset khi lên sau xuống: 6 | đúng: 4


**Vì sao sai:** nếu không reset, `up` và `down` của hai ngọn núi riêng cộng dồn thành một ngọn núi "giả". **Đúng:** reset khi `(down and lên) or bằng`.

### Lỗi 5: mở rộng từ **mọi** chỉ số (không chỉ đỉnh) → `O(n²)` hoặc sai

Nếu mở rộng sang trái/phải từ mọi `i` (không kiểm tra `i` là đỉnh) thì với mảng tăng dần dài phải duyệt `O(n)` cho mỗi `i` — `O(n²)` — và còn dễ nhận nhầm chỗ không phải núi.

In [27]:
def expand_every_index(arr):
    n, best, work = len(arr), 0, 0
    for i in range(n):
        l = i
        while l > 0 and arr[l - 1] < arr[l]:
            l -= 1
            work += 1
        r = i
        while r < n - 1 and arr[r] > arr[r + 1]:
            r += 1
            work += 1
        if l < i < r:
            best = max(best, r - l + 1)
    expand_every_index.work = work
    return best


big = list(range(3000)) + list(range(3000, 0, -1))
t0 = time.perf_counter(); r1 = expand_every_index(big); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = Solution().longestMountain(big); t2 = time.perf_counter() - t0
print(f"mở rộng từ mọi chỉ số: kết quả {r1}, tổng bước mở rộng {expand_every_index.work:,d}, {t1 * 1000:.1f} ms")
print(f"bộ đếm một lượt      : kết quả {r2}, {t2 * 1000:.2f} ms")
assert r1 == r2 and expand_every_index.work > 10 * len(big)

mở rộng từ mọi chỉ số: kết quả 6000, tổng bước mở rộng 9,000,000, 1057.7 ms
bộ đếm một lượt      : kết quả 6000, 1.07 ms


**Vì sao chậm:** mỗi `i` trên sườn dài đi lại cả sườn. **Đúng:** chỉ mở rộng từ **đỉnh**, hoặc dùng `up`/`down`, hoặc nhảy cóc.

### Lỗi 6 (bonus): hai con trỏ nhưng chỉ `l += 1` sau mỗi núi

Hai con trỏ mà không nhảy cóc (mỗi lần chỉ `l += 1`) thành `O(n²)`; nhảy `l = r` giữ `O(n)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đoạn con dài nhất có hình dạng `tăng rồi giảm`" → **chuỗi bước tăng/giảm** và quy hoạch `up`/`down` (hoặc bộ đếm một lượt).

**Mẹo:**

1. Chuyển mảng sang **chuỗi bước** `^`, `v`, `=`: cấu trúc hiện rõ.
2. Hai mảng `up[i]`, `down[i]` rồi `up + down + 1`: mẫu quy hoạch "kết thúc tại / bắt đầu tại".
3. Cần Follow up `O(1)` bộ nhớ → nén hai mảng thành hai bộ đếm; cần reset đúng lúc.
4. Hai con trỏ **nhảy cóc**: nhảy tới điểm kết thúc đoạn vừa duyệt, không `+1`.
5. Số phần tử `= số bước + 1`.
6. Luôn có oracle "kiểm mọi mảng con theo định nghĩa" cho mảng nhỏ.

**Họ bài cùng pattern:** Valid Mountain Array (941), Peak Index in a Mountain Array (852), Longest Turbulent Subarray (978), Longest Increasing Subsequence (300, khác: dãy con), Maximum Subarray (53).

## 15. Biến thể

1. **Thung lũng dài nhất** (giảm rồi tăng): đảo dấu hoặc đảo bước.
2. **Đếm số ngọn núi cực đại** (không mở rộng được nữa) — bài tập 5 một phần.
3. **Trả về vị trí** `(l, r)` của ngọn núi dài nhất.
4. **Cho phép đoạn phẳng** ở đỉnh (non-strict hoặc plateau ở đỉnh): thay điều kiện reset.
5. **Mảng tròn** (cuối nối đầu).

Cài đặt biến thể 1 và 3 và đối chiếu:

In [28]:
def longest_valley(arr):
    """Biến thể 1: thung lũng dài nhất = ngọn núi dài nhất của mảng đã đổi dấu."""
    return Solution().longestMountain([-v for v in arr])


def longest_mountain_span(arr):
    """Biến thể 3: (l, r) của một ngọn núi dài nhất, hoặc None."""
    n = len(arr)
    best = None
    i = 1
    while i < n - 1:
        if arr[i - 1] < arr[i] > arr[i + 1]:
            l = i
            while l > 0 and arr[l - 1] < arr[l]:
                l -= 1
            r = i
            while r < n - 1 and arr[r] > arr[r + 1]:
                r += 1
            if best is None or r - l > best[1] - best[0]:
                best = (l, r)
            i = r
        else:
            i += 1
    return best


print("thung lũng dài nhất của", ex1, ":", longest_valley(ex1), "(ví dụ [4,7,3,2,5] không, [7,3,2,5]? kiểm bằng oracle)")
span = longest_mountain_span(ex1)
print("núi dài nhất ở", span, "->", ex1[span[0]:span[1] + 1])
assert span == (1, 5) and longest_mountain_span(ex2) is None
rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(0, 3) for _ in range(rng.randint(1, 12))]
    sp = longest_mountain_span(a)
    assert (sp is None and oracle(a) == 0) or (sp and is_mountain(a[sp[0]:sp[1] + 1]) and sp[1] - sp[0] + 1 == oracle(a))
    assert longest_valley(a) == oracle([-v for v in a])
print("biến thể: vị trí hợp lệ và độ dài = đáp án; thung lũng = oracle trên mảng đổi dấu")

thung lũng dài nhất của [2, 1, 4, 7, 3, 2, 5] : 4 (ví dụ [4,7,3,2,5] không, [7,3,2,5]? kiểm bằng oracle)
núi dài nhất ở (1, 5) -> [1, 4, 7, 3, 2]
biến thể: vị trí hợp lệ và độ dài = đáp án; thung lũng = oracle trên mảng đổi dấu


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mở rộng từ mỗi đỉnh | Từ đỉnh đi xuống hai bên chân | `O(n)` | `O(1)` | Dễ nghĩ nhất; kiểm tra đỉnh trước |
| 2. Hai mảng `up`/`down` | DP "tăng kết thúc tại / giảm bắt đầu tại" | `O(n)` | `O(n)` | Dễ chứng minh, dễ mở rộng |
| 3. Hai con trỏ nhảy cóc | Chân → đỉnh → chân, nhảy tới cuối sườn xuống | `O(n)` | `O(1)` | Một lượt; Follow up |
| 4. Bộ đếm `up`/`down` | Máy trạng thái 3 trạng thái | `O(n)` | `O(1)` | **Đáp án chính**, ngắn nhất |

Cả bốn khớp oracle trên 15 300 mảng vét cạn và hàng nghìn mảng ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [29]:
for a in ([1, 2, 3, 2, 1], [1, 2, 3, 3, 2, 1], [2, 1], [0, 2, 0, 2, 0, 2, 0]):
    print(a, "->", steps(a), "->", Solution().longestMountain(a))

[1, 2, 3, 2, 1] -> ^^vv -> 5
[1, 2, 3, 3, 2, 1] -> ^^=vv -> 0
[2, 1] -> v -> 0
[0, 2, 0, 2, 0, 2, 0] -> ^v^v^v -> 3


### Bài 2 (tính tay)

`arr = [0, 3, 2, 1, 4, 6, 5, 5, 7, 3, 1]`.

- (a) Viết chuỗi bước.
- (b) Liệt kê các ngọn núi cực đại (đoạn `^+ v+`) và độ dài từng cái.
- (c) Đáp án là bao nhiêu? Chạy `trace_counters` để kiểm.

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện bộ đếm một lượt:

In [30]:
def counters_todo(arr):
    up = down = ans = 0
    for i in range(1, len(arr)):
        # TODO 1: reset up, down khi (down > 0 và arr[i-1] < arr[i]) hoặc arr[i-1] == arr[i]
        # TODO 2: tăng up nếu lên, tăng down nếu xuống
        # TODO 3: nếu up > 0 và down > 0: cập nhật ans = max(ans, up + down + 1)
        pass
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert counters_todo([2, 1, 4, 7, 3, 2, 5]) == 5 and counters_todo([2, 2, 2]) == 0 and counters_todo([1, 2, 1]) == 3

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng hai mảng `up`/`down`:

In [31]:
class MySolution:
    def longestMountain(self, arr):
        pass            # TODO


# Khi làm xong:
# assert MySolution().longestMountain([2, 1, 4, 7, 3, 2, 5]) == 5 and MySolution().longestMountain([2, 2, 2]) == 0

### Bài 5 (mini challenge): đếm số ngọn núi cực đại

Một ngọn núi **cực đại** là mảng con núi không thể mở rộng thêm sang trái hay phải mà vẫn là núi. Viết `count_maximal_mountains(arr)` trả về số ngọn núi cực đại, trong một lượt `O(n)`. (Mỗi đỉnh `arr[i−1] < arr[i] > arr[i+1]` tạo đúng một ngọn núi cực đại.) Kiểm bằng cách liệt kê mọi mảng con là núi và đếm những cái không mở rộng được.

In [32]:
def count_maximal_mountains(arr):
    pass            # TODO


# Khi làm xong:
# assert count_maximal_mountains([2, 1, 4, 7, 3, 2, 5]) == 1 and count_maximal_mountains([1, 2, 1, 2, 1]) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: viết chuỗi bước.
- Mức 2: `[1,2,3,2,1]` → `^^vv` → 5; `[1,2,3,3,2,1]` có `=` → mỗi nửa chưa đủ.
- Mức 3: `[1,2,3,3,2,1]` có bước `=` ở giữa nên không có đỉnh nào có cả hai sườn (`[1,2,3]` thiếu sườn xuống, `[3,2,1]` thiếu sườn lên) → `0`; `[2,1]` quá ngắn → `0`; `[0,2,0,2,0,2,0]` mỗi núi `[0,2,0]` dài `3`. Kết quả `5, 0, 0, 3`.

**Bài 2**

- Mức 1: so sánh từng cặp kề.
- Mức 2: bước: `^ v v ^ ^ v = ^ v v`.
- Mức 3: các đoạn `^+ v+`: `[0,3,2,1]` (4), `[1,4,6,5]` (4), `[5,7,3,1]` (4); chú ý `[6,5,5]` có `=`. Đáp án `4`.

**Bài 3**

- Mức 1: giống mục 9.
- Mức 2: reset khi `(down and a < b) or a == b`.
- Mức 3: `if up and down: ans = max(ans, up + down + 1)`.

**Bài 4**

- Mức 1: `up[i] = up[i-1] + 1` nếu `arr[i-1] < arr[i]`.
- Mức 2: `down[i] = down[i+1] + 1` nếu `arr[i] > arr[i+1]`.
- Mức 3: `max(up[i] + down[i] + 1 for i if up[i] and down[i])`.

**Bài 5**

- Mức 1: mỗi ngọn núi cực đại có đúng một đỉnh.
- Mức 2: đếm `i` với `arr[i-1] < arr[i] > arr[i+1]`.
- Mức 3: kiểm bằng cách đếm các mảng con là núi mà không thể mở rộng (thêm phần tử hai phía vẫn là núi).

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [33]:
res = [Solution().longestMountain(a) for a in ([1, 2, 3, 2, 1], [1, 2, 3, 3, 2, 1], [2, 1], [0, 2, 0, 2, 0, 2, 0])]
assert res == [5, 0, 0, 3]
assert [oracle(a) for a in ([1, 2, 3, 2, 1], [1, 2, 3, 3, 2, 1], [2, 1], [0, 2, 0, 2, 0, 2, 0])] == res
print("Bài 1 OK:", res)

Bài 1 OK: [5, 0, 0, 3]


### Bài 2

(a) `^ v v ^ ^ v = ^ v v`. (b) Các núi cực đại: `[0,3,2,1]` (4), `[1,4,6,5]` (4), `[5,7,3,1]` (4) (bước `=` giữa `5,5` phá núi `[1,4,6,5,5]`). (c) Đáp án `4`.

In [34]:
a = [0, 3, 2, 1, 4, 6, 5, 5, 7, 3, 1]
assert steps(a) == "^vv^^v=^vv"
assert Solution().longestMountain(a) == oracle(a) == 4
for l, r in ((0, 3), (3, 6), (7, 10)):
    assert is_mountain(a[l:r + 1]) and r - l + 1 == 4
print("Bài 2 OK: steps =", steps(a), "-> đáp án", Solution().longestMountain(a))

Bài 2 OK: steps = ^vv^^v=^vv -> đáp án 4


### Bài 3

In [35]:
def counters_answer(arr):
    up = down = ans = 0
    for i in range(1, len(arr)):
        if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
            up = down = 0
        if arr[i - 1] < arr[i]:
            up += 1
        elif arr[i - 1] > arr[i]:
            down += 1
        if up and down:
            ans = max(ans, up + down + 1)
    return ans


assert counters_answer(ex1) == 5 and counters_answer(ex2) == 0 and counters_answer([1, 2, 1]) == 3
rng = random.Random(3)
for _ in range(500):
    a = [rng.randint(0, 3) for _ in range(rng.randint(1, 10))]
    assert counters_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def longestMountain(self, arr):
        n = len(arr)
        up, down = [0] * n, [0] * n
        for i in range(1, n):
            if arr[i - 1] < arr[i]:
                up[i] = up[i - 1] + 1
        for i in range(n - 2, -1, -1):
            if arr[i] > arr[i + 1]:
                down[i] = down[i + 1] + 1
        return max((up[i] + down[i] + 1 for i in range(n) if up[i] and down[i]), default=0)


assert MySolutionAnswer().longestMountain(ex1) == 5 and MySolutionAnswer().longestMountain(ex2) == 0
rng = random.Random(4)
for _ in range(500):
    a = [rng.randint(0, 3) for _ in range(rng.randint(1, 10))]
    assert MySolutionAnswer().longestMountain(a) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Mỗi đỉnh `arr[i−1] < arr[i] > arr[i+1]` thuộc đúng một ngọn núi cực đại (và ngược lại); đếm đỉnh.

In [37]:
def count_maximal_mountains_answer(arr):
    return sum(1 for i in range(1, len(arr) - 1) if arr[i - 1] < arr[i] > arr[i + 1])


def count_maximal_oracle(arr):
    """Đếm mảng con là núi mà không thể mở rộng sang trái hoặc phải (vẫn là núi)."""
    n = len(arr)
    cnt = 0
    for l in range(n):
        for r in range(l + 2, n):
            if not is_mountain(arr[l:r + 1]):
                continue
            left_ext = l > 0 and is_mountain(arr[l - 1:r + 1])
            right_ext = r < n - 1 and is_mountain(arr[l:r + 2])
            if not left_ext and not right_ext:
                cnt += 1
    return cnt


assert count_maximal_mountains_answer([2, 1, 4, 7, 3, 2, 5]) == 1 and count_maximal_mountains_answer([1, 2, 1, 2, 1]) == 2
rng = random.Random(5)
for _ in range(500):
    a = [rng.randint(0, 3) for _ in range(rng.randint(1, 10))]
    assert count_maximal_mountains_answer(a) == count_maximal_oracle(a), a
print("Bài 5 OK: khớp oracle liệt kê mọi mảng con trên 500 mảng")

Bài 5 OK: khớp oracle liệt kê mọi mảng con trên 500 mảng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đoạn con dài nhất dạng "tăng rồi giảm", một lượt, `O(1)` bộ nhớ):**

```python
class Solution:
    def longestMountain(self, arr):
        up = down = ans = 0
        for i in range(1, len(arr)):
            if (down and arr[i - 1] < arr[i]) or arr[i - 1] == arr[i]:
                up = down = 0                 # núi cũ kết thúc / đoạn phẳng
            if arr[i - 1] < arr[i]:
                up += 1
            elif arr[i - 1] > arr[i]:
                down += 1
            if up and down:
                ans = max(ans, up + down + 1)  # +1: số phần tử
        return ans
```

**Checklist trước khi submit:**

- [ ] So sánh **chặt** (`<`, `>`); gặp `==` thì reset (đoạn phẳng phá núi)?
- [ ] Cần **cả hai** sườn: `up > 0` và `down > 0`?
- [ ] Độ dài `= up + down + 1` (số phần tử, không phải số bước)?
- [ ] Reset khi **lên sau khi đã xuống** (bắt đầu núi mới với `up = 1`)?
- [ ] Hai con trỏ: **nhảy cóc** `l = max(r, l + 1)`, không `l += 1` mỗi lần?
- [ ] Mảng ngắn (`< 3`) trả `0`?
- [ ] Đã thử `[2,1,4,7,3,2,5]` (5), `[2,2,2]` (0), `[1,3,3,2]` (0), `[1,2,1]` (3), `[1,2,1,2,1]` (3)?
- [ ] Đúng tên `class Solution`, `longestMountain(self, arr)`, không in debug.